In [ ]:
pip install findspark

In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [1]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [2]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
osql_jar_path = r"file:///C:/spark_drivers/ojdbc17.jar"

spark = SparkSession.builder \
    .appName("SQLServerToosqlTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{osql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

Findspark initialized successfully.
Spark session created.


In [6]:
# Paste the following modified code into your active PySpark shell session
import time
# --- Configuration (UPDATE THESE VALUES) ---
# Start the timer
start_time = time.time()
JDBC_URL = "jdbc:sqlserver://HYPE\\SQL_2022:1433;"
SOURCE_DB_NAME = "AdventureSales" 
TARGET_TABLE = "Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint"
USERNAME = "sa"
PASSWORD = "D@1ry$2014"
DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver" 

# Connection Properties for the READ operation (no change needed here)
read_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
}

# Connection Properties for the WRITE operation
# We rely on the default OVERWRITE behavior (DROP and CREATE TABLE)
write_properties = {
    "user": USERNAME,
    "password": PASSWORD,
    "driver": DRIVER,
    "encrypt": "true",
    "trustServerCertificate": "true",
    # Remove "truncate": "true" here
    # --- ADD THIS LINE FOR BULK INSERT SPEED ---
    "bulkCopy": "true" 
}


# --- 1. Define the Source SQL Query (No change) ---
source_sql_query = """
(SELECT 
    f.region, a.branch_id, f.branch_name, a.year, a.month, a.week, c.channel, sub_channel, prodLine, prodgrp, 
    d.udf6 as psub_id, g.depo, g.sp_name, g.sp_id, d.text_id, product, d.udf1 as psub_id1,
    case when c.CRM = 'Y' then 'Y' else 'N' end as CRM,
    case when ((h.sls_id_chl <>'' and y.sls_id<>'') or sls_id_org<>'') and y.no<=100 then 'Y' else 'N' end as Sls_100,
    case when (h.sls_id_chl <>'' and y.sls_id<>'') or sls_id_org<>''  then 'Y' else 'N' end as Sls_200,
    case when u.custd_id<>'' then 'Y' else 'N' end as DW_18,
    case when v.custd_id<>'' then 'Y' else 'N' end as DW_Oth, Retwhs,
    case when c.Chiller = 'Y' then 'Y' else 'N' end as Chiller,
    convert(dec(17,5),sum(a.net_qty)) as net_qty, convert(dec(17,5),sum(a.net_amt)) as net_amt,
    convert(dec(17,5),sum(a.grs_qty)) as grs_qty, convert(dec(17,5),sum(a.grs_amt)) as grs_amt,
    convert(dec(17,5),sum(a.rtn_qty)) as rtn_qty, convert(dec(17,5),sum(a.rtn_amt)) as rtn_amt,
    convert(dec(17,5),sum(a.grs_amt-a.net_amt-a.rtn_amt)) as disc_amt,
    convert(dec(17,5),sum(i.disc_b_amt)) as Disc_B, convert(dec(17,5),sum(j.disc_PotTunai)) as Disc_PT, 
    case when x.sls_type like '%canva%' then 'Y' else 'N' end  as Canvass,
    case when c.Mkt = 'Y' then 'Y' else 'N' end  as Mkt
FROM sttwk1b a  
LEFT OUTER JOIN customerdist_1910a c ON a.custd_id=c.kode_iap
LEFT OUTER JOIN product_svr55 d ON a.prod_id=d.prod_id
LEFT OUTER JOIN prodcat e ON d.psub_id=e.psub_id
LEFT OUTER JOIN branch f ON a.branch_id=f.branch_id
LEFT OUTER JOIN stockpoint g ON a.sp_id=g.sp_id
LEFT OUTER JOIN discount_b i ON a.custd_id=i.custd_id AND a.prod_id=i.prod_id AND a.year=i.year AND a.week=i.week AND a.sls_id=i.sls_id
LEFT OUTER JOIN discount_potTunai j ON a.custd_id=j.custd_id AND a.prod_id=j.prod_id AND a.year=j.year AND a.week=j.week AND a.sls_id=j.sls_id
LEFT OUTER JOIN sls_chlprn h ON a.branch_id=h.br_chl AND a.sp_id=h.sp_chl AND a.sls_id=h.sls_id_chl AND a.year=h.year AND a.week=h.week
LEFT OUTER JOIN slsmanexc y ON h.br_chl=y.branch_id AND h.sp_chl=y.sp_id AND h.sls_id_prn=y.sls_id 
LEFT OUTER JOIN slsexc_chlprn6 b ON a.sp_id=b.sp_id1 AND a.sls_id=b.sls_id1 AND a.year=b.year AND a.week=b.week
LEFT OUTER JOIN dw_ilo_18 u ON a.custd_id=u.custd_id
LEFT OUTER JOIN dw_ilo_oth v ON a.custd_id=v.custd_id
LEFT OUTER JOIN slsman x ON a.branch_id=x.branch_id AND a.sp_id=x.sp_id AND a.sls_id=x.sls_id
GROUP BY 
    f.region, a.branch_id, f.branch_name, a.year, a.month, a.week, g.depo, g.sp_name, g.sp_id, crm, chiller,
    c.channel, sub_channel, prodLine, prodgrp, d.udf6, d.text_id, product, a.custd_id, d.udf1,
    u.custd_id, v.custd_id, x.sls_type, sls_id_chl, y.sls_id, RetWhs, i.sls_id, j.sls_id, sls_id_org, mkt, y.no
) as custom_query_alias
"""

# --- 2. Load data from SQL Server into a PySpark DataFrame ---
source_df = spark.read.jdbc(
    url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
    table=source_sql_query,
    properties=read_properties
)

print(f"Loaded {source_df.count()} records from source SQL query into PySpark DataFrame.")
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds") 

# --- 3. Overwrite the Target Table (DROP and CREATE TABLE approach) ---
# Start the timer
start_time = time.time()
optimized_df = source_df.repartition(16)
try:
    # Use the optimized DataFrame
    optimized_df.write \
        .mode("overwrite") \
        .jdbc(
            url=f"{JDBC_URL}databaseName={SOURCE_DB_NAME}",
            table=TARGET_TABLE, 
            properties=write_properties # Make sure you use the bulkCopy properties
        )
    print(f"Successfully wrote data with increased parallelism.")
except Exception as e:
    print(f"An error occurred during write operation: {e}")
    
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for load: {total_time:.2f} seconds") 

Loaded 26827257 records from source SQL query into PySpark DataFrame.
Total time taken for load: 48.28 seconds
An error occurred during write operation: An error occurred while calling o57.jdbc.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 9 in stage 23.0 failed 1 times, most recent failure: Lost task 9.0 in stage 23.0 (TID 201) (HYPE executor driver): com.microsoft.sqlserver.jdbc.SQLServerException: The instance of the SQL Server Database Engine cannot obtain a LOCK resource at this time. Rerun your statement when there are fewer active users. Ask the database administrator to check the lock and memory configuration for this instance, or to check for long-running transactions.
	at com.microsoft.sqlserver.jdbc.SQLServerException.makeFromDatabaseError(SQLServerException.java:278)
	at com.microsoft.sqlserver.jdbc.SQLServerStatement.getNextResult(SQLServerStatement.java:1788)
	at com.microsoft.sqlserver.jdbc.SQLServerPreparedStatement.doExecutePreparedStatemen